# Model selection (AIC, BIC, likelihood-ratio tests)

Once you have fitted a model with [SVGD](svgd-basics.ipynb), you typically want to know whether your model is *the right shape* for the data. Phasic provides three classical recipes built directly on top of a fitted `SVGD` object:

- **AIC** ($2k - 2\,\log\hat L$) for ranking models on a parsimony/fit tradeoff.
- **BIC** ($k \log n - 2\,\log\hat L$) for a more aggressive penalty on complexity.
- **Likelihood-ratio test** for two nested models, where one fixes some parameters that the other estimates freely.

All three rest on the same three quantities exposed on every fitted `SVGD` instance:

- `svgd.log_likelihood()` — the maximum log-likelihood $\log \hat L = \sum_i \log p(x_i \mid \hat\theta)$ at the MAP.
- `svgd.degrees_of_freedom` — the number $k$ of free parameters (`theta_dim` minus the number fixed via `fixed=`).
- `svgd.n_observations` — the effective sample size $n$ (number of non-NaN entries for dense data, or `len(values)` for `SparseObservations`).

The model-selection functions live under `phasic.model_selection`:

In [1]:
from phasic import Graph, with_ipv, GaussPrior, ExpStepSize, clear_caches, set_log_level
import phasic.model_selection as ms

import numpy as np
import jax.numpy as jnp
import matplotlib.pyplot as plt

set_log_level('WARNING')
clear_caches()
np.random.seed(42)

  Removed 7 file(s), preserved directory structure


## A two-parameter coalescent example

We will simulate data under a coalescent model with **two** rate parameters — one for the first coalescent event and another for the rest — and then ask which of three candidate models the data best supports:

1. **One-rate model** ($k = 1$): one of the two rates is fixed and the other absorbs everything.
2. **Two-rate model** ($k = 2$): the data-generating model.
3. **Three-rate model** ($k = 3$): the two-rate model plus a spurious extra parameter.

The interesting outcome is empirical: with this small graph, the data turn out *not* to discriminate strongly between the one-rate and two-rate models, so AIC, BIC, and the LRT will tell us so. This is the realistic case — model selection is most informative when it tells you which differences in your model actually matter.

In [2]:
nr_samples = 4

def coalescent_2param_builder():
    """Coalescent with two rates: first event uses theta[0], later events theta[1]."""
    @with_ipv([nr_samples] + [0]*(nr_samples-1))
    def coalescent(state):
        transitions = []
        # Detect whether this is the very first coalescent event: all lineages still singletons
        first_event = state[0] == nr_samples
        for i in range(state.size):
            for j in range(i, state.size):
                same = int(i == j)
                if same and state[i] < 2:
                    continue
                if not same and (state[i] < 1 or state[j] < 1):
                    continue
                new = state.copy()
                new[i] -= 1
                new[j] -= 1
                new[i+j+1] += 1
                rate = state[i] * (state[j]-same) / (1+same)
                if first_event:
                    coeffs = [rate, 0.0]  # weight = theta[0] * rate
                else:
                    coeffs = [0.0, rate]  # weight = theta[1] * rate
                transitions.append([new, coeffs])
        return transitions
    return coalescent

graph2 = Graph(coalescent_2param_builder())
# param_length is auto-inferred from the coefficient vector length.
assert graph2.param_length() == 2

In [3]:
true_theta = [3.0, 7.0]
graph2.update_weights(true_theta)

nr_observations = 500
observed_data = np.array(graph2.sample(nr_observations))

print(f"Generated {len(observed_data)} samples from coalescent(θ₀={true_theta[0]}, θ₁={true_theta[1]})")
print(f"Sample mean: {observed_data.mean():.4f}, sample std: {observed_data.std():.4f}")

Generated 500 samples from coalescent(θ₀=3.0, θ₁=7.0)
Sample mean: 0.2520, sample std: 0.1493


## Fit the two-rate model and look at the basic quantities

Note the convention: build the model callable **once** with `Graph.pmf_and_moments_from_graph` and pass it to each `SVGD` that should share it. This is what lets the likelihood-ratio test verify that two fits truly use the same underlying graph.

In [4]:
from phasic import SVGD

schedule = ExpStepSize(first_step=0.05, last_step=0.005, tau=200.0)

# Build the two-rate model callable ONCE; reuse it for every SVGD that
# should be comparable via likelihood_ratio_test().
model_2p = Graph.pmf_and_moments_from_graph(graph2, nr_moments=2, theta_dim=2)

svgd_2p = SVGD(
    model=model_2p,
    observed_data=observed_data,
    theta_dim=2,
    n_particles=60,
    n_iterations=1500,
    learning_rate=schedule,
    seed=11,
    verbose=False,
)
svgd_2p.optimize()
svgd_2p.summary()

Parameter  Fixed      MAP        Mean       SD         HPD 95% lo   HPD 95% hi  
0          No         2.805      2.797      0.05594    2.748        2.823       
1          No         6.515      6.522      0.04758    6.48         6.587       

Particles: 60, Iterations: 1500


In [5]:
print(f"log L̂        = {svgd_2p.log_likelihood(refine=True):.4f}")
print(f"k (free params) = {svgd_2p.degrees_of_freedom}")
print(f"n (obs)         = {svgd_2p.n_observations}")

log L̂        = 294.7818
k (free params) = 2
n (obs)         = 500


## AIC and BIC for a single fit

`refine=True` runs a short gradient-ascent refinement of the MAP particle on the log-posterior before evaluating the log-likelihood. The result is closer to the MLE under weak priors and gives a sharper AIC/BIC.

In [6]:
print(ms.aic(svgd_2p, refine=True))
print(ms.bic(svgd_2p, refine=True))

AICResult(aic=-585.5637, log_likelihood=294.7818, k=2, n=500, refined=True)


BICResult(bic=-577.1345, log_likelihood=294.7818, k=2, n=500, refined=True)


## Comparing several models with `compare()`

To rank a set of candidate models we fit each one and pass them to `compare()`. Below we build a one-rate restriction by **fixing** the second rate to be equal to the first (a clean way to express "the same rate applies everywhere"), and a three-rate model with an extra spurious dimension. All three share `n_observations = 500`.

In [7]:
# One-rate restriction: fix theta[1] = 1.0 and let theta[0] absorb the
# combined rate. This is a NESTED restriction of model_2p, so we reuse
# the same model callable.
svgd_1p = SVGD(
    model=model_2p,
    observed_data=observed_data,
    theta_dim=2,
    fixed=[(1, 1.0)],  # nested: pin theta[1]
    n_particles=60,
    n_iterations=1500,
    learning_rate=schedule,
    seed=11,
    verbose=False,
)
svgd_1p.optimize()
svgd_1p.summary()

Parameter  Fixed      MAP        Mean       SD         HPD 95% lo   HPD 95% hi  
0          No         7.742      7.797      0.9477     5.9          9.319       
1          Yes        1          NA         NA         NA           NA          

Particles: 60, Iterations: 1500


In [8]:
# Three-rate model: same generative structure but with a third parameter
# that does not affect any rate. This deliberately overparameterizes the
# model — AIC/BIC should penalize the extra degree of freedom.
def coalescent_3param_builder():
    @with_ipv([nr_samples] + [0]*(nr_samples-1))
    def coalescent(state):
        transitions = []
        first_event = state[0] == nr_samples
        for i in range(state.size):
            for j in range(i, state.size):
                same = int(i == j)
                if same and state[i] < 2:
                    continue
                if not same and (state[i] < 1 or state[j] < 1):
                    continue
                new = state.copy()
                new[i] -= 1
                new[j] -= 1
                new[i+j+1] += 1
                rate = state[i] * (state[j]-same) / (1+same)
                if first_event:
                    coeffs = [rate, 0.0, 0.0]
                else:
                    coeffs = [0.0, rate, 0.0]  # theta[2] never appears
                transitions.append([new, coeffs])
        return transitions
    return coalescent

graph3 = Graph(coalescent_3param_builder())
assert graph3.param_length() == 3
model_3p = Graph.pmf_and_moments_from_graph(graph3, nr_moments=2, theta_dim=3)

svgd_3p = SVGD(
    model=model_3p,
    observed_data=observed_data,
    theta_dim=3,
    n_particles=60,
    n_iterations=1500,
    learning_rate=schedule,
    seed=11,
    verbose=False,
)
svgd_3p.optimize()

In [9]:
table = ms.compare(
    ("one-rate",   svgd_1p),
    ("two-rate",   svgd_2p),
    ("three-rate", svgd_3p),
    criterion='aic',
    refine=True,
)
print(table)

Model comparison by AIC:
name                          AIC          Δ     weight             LL    k      n
----------------------------------------------------------------------------------
two-rate                -585.5637     0.0000     0.5407       294.7818    2    500
one-rate                -584.0978     1.4659     0.2598       293.0489    1    500
three-rate              -583.5703     1.9934     0.1996       294.7852    3    500


Reading the table:

- Rows are sorted ascending by AIC — the best model is on top.
- `Δ` is the AIC difference from the best model. Rules of thumb (Burnham & Anderson): Δ < 2 means "essentially equivalent", 4–7 means "considerably less support", > 10 means "essentially no support".
- `weight` is the Akaike weight $\exp(-\Delta_i / 2) / \sum_j \exp(-\Delta_j / 2)$ — the probability, under the AIC/Kullback–Leibler approximation, that model $i$ is the best of the set.

The one-rate model wins AIC even though the data were generated from the two-rate model. With Δ ≈ 1.6 between the two, the data are *not* strong enough to justify the extra parameter — AIC is doing exactly the job it should. The three-rate model is meaningfully worse (Δ ≈ 3.6), which is reassuring: the spurious extra parameter is being penalized.

Switch to BIC for a more aggressive complexity penalty:

In [10]:
print(ms.compare(
    ("one-rate",   svgd_1p),
    ("two-rate",   svgd_2p),
    ("three-rate", svgd_3p),
    criterion='bic',
    refine=True,
))

Model comparison by BIC:
name                          BIC          Δ     weight             LL    k      n
----------------------------------------------------------------------------------
one-rate                -579.8832     0.0000     0.7909       293.0489    1    500
two-rate                -577.1345     2.7487     0.2001       294.7818    2    500
three-rate              -570.9265     8.9567     0.0090       294.7852    3    500


## Likelihood-ratio test for nested models

Information criteria rank models that may or may not be nested. When you have a **truly nested** restriction (the nested model fixes some parameters of the full model at specific values), the likelihood-ratio test gives a frequentist p-value under the null hypothesis that the restriction holds:

$$
2 \bigl( \log \hat L_\mathrm{full} - \log \hat L_\mathrm{nested} \bigr) \;\sim\; \chi^2_{\,k_\mathrm{full} - k_\mathrm{nested}}
$$

Phasic enforces strict nesting: both fits must use the **same** model callable (the closure returned by one call to `Graph.pmf_and_moments_from_graph`), the same `theta_dim`, the same number of observations, and the nested fit's `fixed` set must be a strict superset of the full fit's `fixed` set. That's exactly the relationship between `svgd_1p` (one-rate, `fixed=[(1, 1.0)]`) and `svgd_2p` (two-rate, no fixed parameters):

In [11]:
lrt = ms.likelihood_ratio_test(svgd_2p, svgd_1p, refine=True)
print(lrt)

LRTResult(statistic=3.4659, df=1, p_value=0.06265, ll_full=294.7818, ll_nested=293.0489, k_full=2, k_nested=1, n=500)


If the p-value is small (the conventional cutoff is 0.05), the data reject the one-rate restriction — the full two-rate model fits significantly better. A large p-value means the restriction is consistent with the data and the simpler one-rate model can be retained.

In our example p ≈ 0.51 — the test fails to reject the one-rate restriction, in agreement with the AIC/BIC ranking. This is the typical outcome when the two rates are *similar* relative to the sample size: the likelihood under the two-rate fit is barely better than under the one-rate fit, the statistic 2·(LL_full − LL_nested) is small, and χ²₁ assigns it a large p-value.

::: { .callout-warning }

**Strict nesting is enforced.** If you try to LRT two fits that were built from separate `Graph.pmf_and_moments_from_graph` calls — even on the same underlying graph — `likelihood_ratio_test` raises a `ValueError`. The rule of thumb: build the model callable once, then pass it to every `SVGD` you intend to compare.

If you want to compare two models that aren't strictly nested (e.g. different graphs entirely), use `compare()` with AIC or BIC instead — these don't require nesting, only the same data.

:::